In [14]:
%%capture
#Load unsloth
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [15]:
#Re-installation to solve the errors
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install --upgrade --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth-zoo.git

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-l53a4igm
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-l53a4igm
  Resolved https://github.com/unslothai/unsloth.git to commit de21723a0daaed3268bbbe78d781802afb577831
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.5.10-py3-none-any.whl size=34803329 sha256=a21a4c4c4b2eda822445217ae581f7df6d830dd60f7697d0f5cf7d16f032bd95
  Stored in directory: /tmp/pip-ephem-wheel-cache-y0ynuupe/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2026.5.10
    Uninstalling unsloth-2026.5.10:
      Successfully uninstalled unsloth-2026.5.10
  Cloning https://github.com/unslothai/unsloth-zoo.git to /tmp/pip-req-build

In [16]:
#Upgrading huggingface and transformers
!pip install --upgrade --no-cache-dir --no-deps transformers huggingface_hub

In [17]:
# RAG dependencies
!pip install wikipedia-api duckduckgo-search beautifulsoup4 scikit-learn
# Speech mode dependencies
!pip install faster-whisper librosa soundfile

In [18]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [117]:
%%capture
!pip install --upgrade jupyter-client

In [127]:
import warnings
# Suppress DeprecationWarnings from third-party packages (e.g. jupyter_client utcnow)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [128]:
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

In [129]:
import os
os.environ["PYTHONWARNINGS"] = "ignore::DeprecationWarning"

In [132]:
import warnings, logging, os

# Silence every warning
warnings.filterwarnings("ignore")                          # all Python warnings
os.environ["PYTHONWARNINGS"] = "ignore"                    # subprocess warnings
logging.getLogger("transformers").setLevel(logging.ERROR)  # transformers chatter
logging.getLogger("unsloth").setLevel(logging.ERROR)

In [22]:
#Model loading
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 5.9.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

unsloth/Qwen2.5-14B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [155]:
#PEFT/LoRA Configuration
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [23]:
# Preparing for fast inference
FastLanguageModel.for_inference(model)


# ============================================================
# SPEECH-TO-TEXT MODEL  (local Whisper — obeying the rules)
# Transcribes the game's spoken questions. It runs locally,
# so the "models must run locally" rule is respected.
# ============================================================
from faster_whisper import WhisperModel

WHISPER_MODEL = "deepdml/faster-whisper-large-v3-turbo-ct2"
whisper_model = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="int8")
print("Whisper loaded — listen, now I can.")


Whisper loaded — listen, now I can.


In [1]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

# Canonical persistent log location on Drive — survives runtime disconnects
DRIVE_DIR = "/content/gdrive/MyDrive/Colab Notebooks"
os.makedirs(DRIVE_DIR, exist_ok=True)
LOG_FILE = os.path.join(DRIVE_DIR, "game_log.json")
print(f"LOG_FILE = {LOG_FILE}")

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).
LOG_FILE = /content/gdrive/MyDrive/Colab Notebooks/game_log.json


Then we need to add our python package "millionaire_client" to the system path, so python can see it.

In [2]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment_api_client'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/Colab Notebooks/NLP_assignment_api_client']


Let's import the client classes

In [3]:
from millionaire_client import MillionaireClient, AuthenticationError

You can save your password in a Colab secret (the "key" icon on the tab on the left) and import it into your notebook.

In [4]:
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

Now keep the API_URL as stated, but please change the username and password to be the ones you used during sign up session.

In [5]:
API_URL = "http://131.175.15.22:51111/"
username = "OmarAzab"
password = pwd

Now we can instantiate a MillionaireClient object and call the login method, which takes as parameters username and password.

In [6]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, OmarAzab! (Role: student)


After login, the web page is showing you different types of competitions, for each of them you can choose to play a game or to see the leaderboard. For now let's list all of the.

In [7]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)
  4: Philosophy and Psychology (15 questions)
  5: News (15 questions)


In [136]:
# Choose a competition ID
comp_id = 1

In [9]:
import spacy
import unicodedata

try:
    _nlp = spacy.load("en_core_web_sm")
except OSError:
    import subprocess
    subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], check=True)
    _nlp = spacy.load("en_core_web_sm")

def build_query_llm(question_text, options):
    formatted = "\n".join(f"[{o.id}] {o.text}" for o in options)
    messages = [
        {"role": "system", "content": (
            "You generate Wikipedia search queries. "
            "Identify the KEY CONCEPT or SUBJECT the question asks about — "
            "not the answer options. "
            "Output ONLY the search term (1-5 words), nothing else. "
            "Never output any answer option text."
        )},
        {"role": "user", "content": (
            f"Question: {question_text}\nOptions:\n{formatted}\n\nSearch term:"
        )},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True,
                                            add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(input_ids=inputs, max_new_tokens=15,
                             temperature=0.1, do_sample=True, use_cache=True)
    query = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip().split("\n")[0][:80]

    # Reject queries that contain an answer option — anchoring guard
    option_texts = {o.text.strip().lower() for o in options}
    query_lower  = query.lower()
    anchored = any(
        opt in query_lower or query_lower in opt
        for opt in option_texts
    )
    if anchored:
        # 1. quoted phrase
        quoted = re.findall(r"['\"\u2018\u2019\u201c\u201d]([^'\"\u2018\u2019\u201c\u201d]+)['\"\u2018\u2019\u201c\u201d]", question_text)
        if quoted:
            query = quoted[0].strip()
        else:
            # 2. "concept/practice/idea of X" — grab X
            concept = re.search(
                r"(?:concept|practice|idea|notion|principle|ritual|ceremony|tradition|theory) of (\w+(?:\s+\w+)?)",
                question_text, re.IGNORECASE
            )
            if concept:
                query = concept.group(1).strip()
            else:
                # 3. longest capitalised multi-word phrase not matching an option
                caps = re.findall(r"\b(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b", question_text)
                caps = [c for c in caps if c.lower() not in option_texts]
                if caps:
                    caps.sort(key=len, reverse=True)
                    query = caps[0]
                else:
                    query = question_text[:80]

    return query

def _strip_accents(text):
    # Convert "émile" → "emile", "Babylṓn" → "Babylon"
    nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in nfkd if not unicodedata.combining(c))

def is_context_relevant(retrieved_title, entities):
    if not entities:
        return True
    title_norm = _strip_accents(re.sub(r"[-'']", " ", retrieved_title.lower().strip()))

    matched_entity = False
    for ent in entities:
        ent_norm = _strip_accents(re.sub(r"[-'']", " ", ent.lower().strip()))
        if ent_norm in title_norm or title_norm in ent_norm:
            matched_entity = True
            break
        if len(ent_norm) >= 5 and ent_norm[:5] in title_norm:
            matched_entity = True
            break

    return matched_entity

def get_entities(question_text):
    doc = _nlp(question_text)
    useful = {"PERSON", "WORK_OF_ART", "ORG", "EVENT", "FAC", "PRODUCT", "GPE", "NORP", "LOC"}
    return [e.text.strip() for e in doc.ents
            if e.label_ in useful and len(e.text.strip()) > 2]


## Speech-to-Text

When the game runs in speech mode, it reads the question and options aloud. We must transcribe that audio before the model can answer.

Built on local **faster-whisper**, so the assignment's rule — "the models must run locally" — is respected. Whisper sometimes mishears the game's expressive TTS voice; a cleanup step corrects those slips. We give Whisper domain-specific priming vocabulary so it can catch proper nouns like "Parthenon" and "mitochondria".

In [166]:
import io, re, time
import numpy as np
import librosa
from collections import Counter

# Map competition id -> domain name (helps prime Whisper)
COMP_DOMAIN = {0: "entertainment", 1: "history", 2: "science",
               3: "math", 4: "philosophy", 5: "news"}

def domain_of(comp_id):
    return COMP_DOMAIN.get(comp_id, "default")

# Per-domain priming vocabulary — helps Whisper hear proper nouns better
DOMAIN_PROMPTS = {
    "math":          "Spoken clearly. Option A, Option B, Option C, Option D. cyclic group, subgroup, integers, modular arithmetic, probability, standard deviation, correlation, binomial, derivative, integral, matrix, eigenvalue",
    "science":       "Spoken clearly. Option A, Option B, Option C, Option D. mitochondria, lysosome, phytoplankton, endoplasmic reticulum, Golgi apparatus, neutrons, protons, electrons, isotope, ectothermic, photosynthesis",
    "history":       "Spoken clearly. Option A, Option B, Option C, Option D. dynasty, Byzantine, Achaeans, Assyrian, Parthenon, Athenian, Aristotle, Renaissance, Constantinople, Djoser, Khufu, Sneferu, Tawagalawa",
    "entertainment": "Spoken clearly. Option A, Option B, Option C, Option D. directed by, starring, Academy Award, protagonist, antagonist, MacGuffin, cinematography, soundtrack",
    "philosophy":    "Spoken clearly. Option A, Option B, Option C, Option D. Aristotle, Plato, Socrates, Kant, Nietzsche, Hegel, epistemology, metaphysics, ethics, empiricism, rationalism, utilitarianism, Stoicism, Wittgenstein, Bayesian",
    "news":          "Spoken clearly. Option A, Option B, Option C, Option D. president, government, election, policy, conflict, treaty, economy, sanctions, summit, referendum, parliament, diplomacy, ceasefire, inflation",
    "default":       "Spoken clearly. Option A, Option B, Option C, Option D",
}

MAX_TRANSCRIBE_SECONDS = 3.0   # run away the lazy generator must not

# Known mishearings of this TTS voice — fix them
WORD_CORRECTIONS = {
    r'\btongue\b': 'term',          r'\bwhammy\b': 'kwame',
    r'\bcargable\b': 'clark gable', r'\bnervana\b': 'nirvana',
    r'\bjestive\b': 'digestive',    r'\bconencilism\b': 'commensalism',
    r'\bsalamunaya\b': 'sal ammoniac', r'\baxobiology\b': 'astrobiology',
    r'\bpytoskeleton\b': 'cytoskeleton', r'\bhontology\b': 'ontology',
    r'\bthompson\b': 'option',      r'\btopsin\b': 'option',
}

def clean_stt_text(text):
    """Fix known mishearings and strip garbled 'Option X' labels, this does."""
    for pat, repl in WORD_CORRECTIONS.items():
        text = re.sub(pat, repl, text, flags=re.I)
    text = re.sub(r'^(and[,\s]+)?(op[a-z]*\s*[abcd\d][\.,]?\s*|top[a-z\s]+[abcd\d][\.,]?\s*|'
                  r'thompson\s+[abcd\d][\.,]?\s*|topson\s+[abcd\d][\.,]?\s*)',
                  '', text.strip(), flags=re.I)
    text = re.sub(r'\s*(tops?\s+and\s+[abcd\d][\.,]?|thompson\s+[abcd\d][\.,]?)$', '', text, flags=re.I)
    return text.strip()

def _is_hallucination(text, elapsed):
    """Runaway transcription, detect we must — too long, or repeated too much."""
    if elapsed > MAX_TRANSCRIBE_SECONDS:
        return True
    words = text.lower().split()
    return len(words) > 8 and Counter(words).most_common(1)[0][1] / len(words) > 0.6

# Track timings — useful for the report
stt_timings = []

def transcribe_audio(audio_bytes, comp_id=None, context=""):
    """Audio bytes into clean text, turn we must ('' if useless it is)."""
    t0 = time.time()
    domain = domain_of(comp_id)
    is_math = (domain == "math")

    audio, _ = librosa.load(io.BytesIO(audio_bytes), sr=16000)
    audio = audio.astype(np.float32)
    if not is_math:                              # slowly already, math is read
        audio = librosa.effects.time_stretch(audio, rate=0.85)

    prompt = DOMAIN_PROMPTS.get(domain, DOMAIN_PROMPTS["default"])
    if context:
        prompt = f"{prompt}. {context[:80]}"

    segments, _ = whisper_model.transcribe(
        audio, initial_prompt=prompt, language="en",
        condition_on_previous_text=False, temperature=0.0, beam_size=5,
        no_speech_threshold=0.6, log_prob_threshold=-1.0,
        compression_ratio_threshold=2.4, vad_filter=is_math,
    )

    parts = []
    for seg in segments:
        parts.append(seg.text)
        if time.time() - t0 > MAX_TRANSCRIBE_SECONDS:
            break                                # spiral into hallucination, let it not

    text = clean_stt_text(" ".join(parts).strip())
    elapsed = time.time() - t0
    stt_timings.append(elapsed)

    if not is_math and _is_hallucination(text, elapsed):
        return ""                                # fall back to the options, the LLM will
    print(f"  [transcribe {elapsed:.2f}s] {text}")
    return text


In [25]:
import re, time, unicodedata, warnings
import requests
import wikipediaapi
from duckduckgo_search import DDGS
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

TEXT_REFERENCE = re.compile(
    r"\b(according to the (given |above |following )?text|"
    r"according to the passage|based on the (given |above )?text|"
    r"as stated in the text|the text (states|says|suggests|implies)|"
    r"as described in the text)\b",
    re.IGNORECASE
)

WIKI_HEADERS = {
    "User-Agent": "PoliMillionaireBot/1.0 (NLP course project; student@polimi.it)"
}

BLOCKED_DOMAINS = {
    "quizlet.com", "chegg.com", "coursehero.com", "brainly.com",
    "yahoo.com", "christianwebsite.com", "pinterest.com",
    "reddit.com", "facebook.com", "twitter.com", "instagram.com"
}

warnings.filterwarnings("ignore")

# Initialise the Wikipedia client
wiki = wikipediaapi.Wikipedia(
    language   = "en",
    user_agent = "PoliMillionaireBot/1.0 (NLP course assignment)"
)

def clean_text(text):
    text = unicodedata.normalize("NFKD", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def chunk_text(text, max_chars=600):
    # Split the text into sentences
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, buf = [], ""
    for sent in sentences:
        if len(buf) + len(sent) < max_chars:
            buf += " " + sent
        else:
            if buf:
                chunks.append(buf.strip())
            buf = sent
    if buf:
        chunks.append(buf.strip())
    return chunks or [text[:max_chars]]

def best_passage(query, chunks):
    # Find the most relevant chunk
    if not chunks:
        return ""
    if len(chunks) == 1:
        return chunks[0]
    try:
        vec   = TfidfVectorizer(stop_words="english")
        tfidf = vec.fit_transform([query] + chunks)
        sims  = cosine_similarity(tfidf[0:1], tfidf[1:]).flatten()
        return chunks[int(sims.argmax())]
    except Exception:
        return chunks[0]

from sentence_transformers import SentenceTransformer, util
_embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

def passage_relevance_score(question_text, passage):
    try:
        emb = _embedder.encode([question_text, passage], convert_to_tensor=True)
        return float(util.cos_sim(emb[0], emb[1]))
    except Exception:
        return 0.0

def retrieve_wikipedia(query, question_text):
    # --- Step 1: search for the title (with 429 backoff) ---
    title = None
    search_endpoints = [
        f"https://en.wikipedia.org/w/rest.php/v1/search/page?q={requests.utils.quote(query)}&limit=3",
        f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={requests.utils.quote(query)}&srlimit=3&format=json&utf8=1",
    ]
    for url in search_endpoints:
        try:
            resp = requests.get(url, timeout=5, headers=WIKI_HEADERS)
            if resp.status_code == 429:
                time.sleep(2)
                continue
            if not resp.text.strip():
                continue
            data = resp.json()
            hits = data.get("pages") or data.get("query", {}).get("search", [])
            if hits:
                title = hits[0].get("title")
                break
        except Exception as exc:
            print(f"  [RAG-Wiki] Search error: {exc}")
            continue

    if not title:
        return ""

    # --- Step 2: relevance gate ---
    entities = get_entities(question_text)
    entities_plus_query = entities + [query]
    if entities and not is_context_relevant(title, entities_plus_query):
        print(f"  [RAG-Wiki] Relevance gate blocked: '{title}'")
        return ""

    # --- Step 3: fetch full article text (with 429 backoff) ---
    full_text = ""
    full_url = (
        f"https://en.wikipedia.org/w/api.php?action=query&prop=extracts"
        f"&explaintext=1&titles={requests.utils.quote(title)}&format=json&utf8=1"
    )
    for attempt in range(3):
        try:
            full_resp = requests.get(full_url, timeout=6, headers=WIKI_HEADERS)
            if full_resp.status_code == 429:
                print("  [RAG-Wiki] Article fetch rate limited — backing off 2s")
                time.sleep(2)
                continue
            if not full_resp.text.strip():
                time.sleep(1)
                continue
            pages = full_resp.json().get("query", {}).get("pages", {})
            if pages:
                full_text = clean_text(list(pages.values())[0].get("extract", ""))
            break
        except Exception as exc:
            print(f"  [RAG-Wiki] Article fetch error: {exc}")
            time.sleep(1)

    if len(full_text) < 50:
        return ""

    passage = best_passage(question_text, chunk_text(full_text))
    return f"[Wikipedia – {title}]\n{passage}"

def retrieve_duckduckgo(query, question_text):
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))  # increased to 5 to have more to skip
        if not results:
            return ""
        for r in results:
            url = r.get("href", "")
            domain = re.sub(r"https?://(www\.)?", "", url).split("/")[0]
            if not url or "wikipedia" in url or domain in BLOCKED_DOMAINS:
                continue
            try:
                page_resp  = requests.get(url, timeout=5, headers={"User-Agent": "Mozilla/5.0"})
                soup       = BeautifulSoup(page_resp.text, "html.parser")
                paragraphs = [
                    clean_text(p.get_text())
                    for p in soup.find_all("p")
                    if len(p.get_text(strip=True)) > 80
                ]
                if not paragraphs:
                    continue
                passage = best_passage(question_text, paragraphs[:20])
                if passage:
                    domain = re.sub(r"https?://(www\.)?", "", url).split("/")[0]
                    return f"[Web – {domain}]\n{passage}"
            except Exception:
                continue
        snippet = results[0].get("body", "")
        return f"[DDG snippet]\n{snippet}" if snippet else ""
    except Exception as exc:
        print(f"  [RAG-DDG] Error: {exc}")
        return ""

NEWS_KEYWORDS = re.compile(
    r"\b(2024|2025|2026|recent|latest|current|last year|this year|"
    r"election|war|crisis|president|prime minister|CEO|appointed|awarded)\b",
    re.IGNORECASE
)

def retrieve_context(question_text, options, timeout=8.0):
    if TEXT_REFERENCE.search(question_text):
        print("  [RAG] Skipped — references a specific source text")
        return ""

    t0    = time.time()
    query = build_query_llm(question_text, options)
    print(f"  [RAG] Query: {repr(query)}")

    # Check if the query is a PERSON — if so, we trust the article regardless of TF-IDF
    doc = _nlp(question_text)
    person_names = {e.text.strip().lower() for e in doc.ents if e.label_ == "PERSON"}
    query_is_person = any(p in query.lower() or query.lower() in p for p in person_names)

    context = retrieve_wikipedia(query, question_text)
    elapsed = time.time() - t0

    if (not context or NEWS_KEYWORDS.search(question_text)) and elapsed < timeout - 3:
        print("  [RAG] Trying DuckDuckGo fallback…")
        ddg_ctx = retrieve_duckduckgo(query, question_text)
        if ddg_ctx and not context:
            context = ddg_ctx
        elif ddg_ctx and NEWS_KEYWORDS.search(question_text):
            context = ddg_ctx

    elapsed = time.time() - t0
    if context:
        passage = context.split("\n", 1)[1] if "\n" in context else context
        score   = passage_relevance_score(question_text, passage)
        src     = context.split("\n")[0]
        print(f"  [RAG] {src} — {elapsed:.1f}s ({len(context)} chars) relevance={score:.2f}")
        # ← Bypass gate for person-driven queries — the article IS about them
        if score < 0.20 and not query_is_person:
            print("  [RAG] Relevance too low — discarding context")
            return ""
        elif score < 0.20 and query_is_person:
            print("  [RAG] Low score but query is a PERSON — trusting article")
    else:
        print(f"  [RAG] No context found in {elapsed:.1f}s — own knowledge, use we must")
    return context

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [26]:
# ============================================================
# FEW-SHOT LEARNING FROM LOGS
# Build few-shot examples from previously-correct answers in the log.
# This teaches the model the question style/domain without any training.
# ============================================================
import json, os, random, re

N_FEWSHOT = 3   # how many examples to include (set 0 to disable / for zero-shot baseline)

def _norm_q(q):
    return re.sub(r"\s+", " ", q.strip().lower())

def build_fewshot_block(question_text, n=N_FEWSHOT, comp_id=None):
    """Return a string of n correct Q/A examples from the log, excluding the current question."""
    if n <= 0 or not os.path.exists(LOG_FILE):
        return ""
    try:
        with open(LOG_FILE) as f:
            log = json.load(f)
    except Exception:
        return ""

    cur = _norm_q(question_text)
    # Keep only CORRECT answers, dedup by question, never leak the current question
    seen, pool = set(), []
    for e in log:
        if not e.get("correct"):
            continue
        if comp_id is not None and e.get("comp_id") != comp_id:
            continue   # same-category examples only
        qn = _norm_q(e["question"])
        if qn == cur or qn in seen:
            continue
        seen.add(qn)
        pool.append(e)

    if not pool:
        return ""

    sampled = random.sample(pool, min(n, len(pool)))
    blocks = []
    for e in sampled:
        opts = "\n".join(f"[{o['id']}] {o['text']}" for o in e["options"])
        blocks.append(f"Question: {e['question']}\nOptions:\n{opts}\nAnswer: {e['answer_given']}")
    return "Here are some example questions and their correct answers:\n\n" + "\n\n".join(blocks) + "\n\n"


In [191]:
import re, torch
import json, os

SYSTEM_NO_RAG = (
    "You are a contestant on a quiz show. "
    "You will be given a question and four options labeled 0, 1, 2, 3. "
    "Choose the single best answer. "
    "Reply with ONLY the digit of the correct option (0, 1, 2, or 3) and nothing else."
)

SYSTEM_PASS1 = (
    "You are a quiz show contestant. "
    "Reply with ONLY two lines and nothing else:\n"
    "Line 1: the digit of the correct option (0, 1, 2, or 3)\n"
    "Line 2: your confidence as a percentage (0-100)\n"
    "Do not write anything else. Do not explain."
)

# ============================================================
# CATEGORY PIPELINE CONFIG (Colab)
#   History (comp_id == 1) -> few-shot prompting (use history LoRA + few-shot + RAG + memory)
#   Other categories       -> base model + RAG only (no few-shot examples)
# Fine-tuning is weights-level: load the history LoRA adapter ONLY when playing history.
# Few-shot is gated per-question below via FEWSHOT_CATEGORIES.
# ============================================================
FEWSHOT_CATEGORIES = set()   # only competition 1 (Ancient History) uses few-shot
RUN_CONFIG = "model+rag"   # change to "adapter+fewshot" for the second run
USE_MEMORY = True        # off for a fair comparison

_last_decision = {"pass1": None, "pass2": None, "overrode": False}

def _generate(messages, max_new_tokens=16):
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens, max_length=None,
            temperature=0.3, do_sample=True, use_cache=True,
        )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    match = re.search(r"[0-3]", response)
    if match:
        return int(match.group(0))
    print(f"  [LLM] Unexpected output: {repr(response)} — default to 0, we must")
    return 0

def _normalize_q(q):
    return re.sub(r"\s+", " ", q.strip().lower())

def get_previous_correct_text(question_text):
    if not os.path.exists(LOG_FILE):
        return None
    try:
        with open(LOG_FILE) as f:
            log = json.load(f)
    except Exception:
        return None
    for entry in log:
        if _normalize_q(entry["question"]) == _normalize_q(question_text) and entry["correct"]:
            for opt in entry["options"]:
                if opt["id"] == entry["answer_given"]:
                    return opt["text"].strip().lower()
    return None

def get_previous_wrong_texts(question_text):
    if not os.path.exists(LOG_FILE):
        return set()
    try:
        with open(LOG_FILE) as f:
            log = json.load(f)
    except Exception:
        return set()
    wrong = set()
    for entry in log:
        if _normalize_q(entry["question"]) == _normalize_q(question_text) and not entry["correct"]:
            for opt in entry["options"]:
                if opt["id"] == entry["answer_given"]:
                    wrong.add(opt["text"].strip().lower())
    return wrong

def get_llm_answer_no_rag(question_text, options):
    formatted = "\n".join(f"[{o.id}] {o.text}" for o in options)
    messages  = [
        {"role": "system", "content": SYSTEM_NO_RAG},
        {"role": "user",   "content": (
            f"Question: {question_text}\nOptions:\n{formatted}\n\nWhat is the correct option ID?"
        )},
    ]
    return _generate(messages)

def get_llm_answer(question_text, options, comp_id=1):
    global _last_decision
    _last_decision = {"pass1": None, "pass2": None, "overrode": False}
    formatted = "\n".join(f"[{o.id}] {o.text}" for o in options)

    # ================= MEMORY LAYER (only when USE_MEMORY is on) =================
    # 1. If we answered this exact question correctly before, repeat that answer immediately.
    prev_wrong = set()
    if USE_MEMORY:
        prev_correct = get_previous_correct_text(question_text)
        if prev_correct:
            for i, o in enumerate(options):
                if o.text.strip().lower() == prev_correct:
                    print(f"  [Memory] Previously correct: '{prev_correct}' — using option {i}")
                    _last_decision = {"pass1": i, "pass2": i, "overrode": False}
                    return i
        prev_wrong = get_previous_wrong_texts(question_text)

    # ================= PASS 1: answer + confidence =================
    # Few-shot ONLY for history (comp_id in FEWSHOT_CATEGORIES).
    # Science/Philosophy/etc -> base model + RAG only, no few-shot examples.
    if comp_id in FEWSHOT_CATEGORIES:
        fewshot = build_fewshot_block(question_text, comp_id=comp_id)
    else:
        fewshot = ""
    messages_pass1 = [
        {"role": "system", "content": SYSTEM_PASS1},
        {"role": "user", "content": (
            f"{fewshot}"
            f"Question: {question_text}\nOptions:\n{formatted}\n\n"
            f"Digit (0-3):\nConfidence (LOW/MEDIUM/HIGH):"
        )},
    ]
    inputs = tokenizer.apply_chat_template(
        messages_pass1, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs, max_new_tokens=12, max_length=None,
            temperature=0.3, do_sample=True, use_cache=True,
        )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    print(f"  [Pass 1 raw]:", response)

    lines        = response.strip().split("\n")
    answer_match = re.search(r"[0-3]", lines[0]) if len(lines) > 0 else None
    conf_match   = re.search(r"\d+", lines[1]) if len(lines) > 1 else None
    first_pass_answer = int(answer_match.group(0)) if answer_match else 0
    confidence        = int(conf_match.group(0))   if conf_match   else 50
    print(f"  [Pass 1] Answer: {first_pass_answer}  Confidence: {confidence}%")

    # Helper: apply elimination of known-wrong answers to a proposed answer.
    def apply_elimination(answer):
        if not prev_wrong:
            return answer
        if options[answer].text.strip().lower() not in prev_wrong:
            return answer
        alternatives = [i for i, o in enumerate(options)
                        if o.text.strip().lower() not in prev_wrong]
        if not alternatives:
            return answer
        if len(alternatives) == 1:
            print(f"  [Memory] '{options[answer].text}' failed before — only option {alternatives[0]} remains")
            return alternatives[0]
        # Re-ask the model among the remaining options
        alt_text = "\n".join(f"[{i}] {options[i].text}" for i in alternatives)
        msg = [
            {"role": "system", "content": (
                "Choose the single best answer from the given options. "
                "Reply with ONLY one digit and nothing else."
            )},
            {"role": "user", "content": f"Question: {question_text}\nOptions:\n{alt_text}\n\nBest option digit:"},
        ]
        ins = tokenizer.apply_chat_template(msg, tokenize=True,
                add_generation_prompt=True, return_tensors="pt").to("cuda")
        with torch.inference_mode():
            o = model.generate(input_ids=ins, max_new_tokens=4, max_length=None,
                               temperature=0.3, do_sample=True, use_cache=True)
        resp = tokenizer.decode(o[0][ins.shape[1]:], skip_special_tokens=True).strip()
        mm = re.search(r"[0-3]", resp)
        chosen = int(mm.group(0)) if (mm and int(mm.group(0)) in alternatives) else alternatives[0]
        print(f"  [Memory] '{options[answer].text}' failed before — re-chose option {chosen} from {alternatives}")
        return chosen

    # ================= Skip RAG if confident =================
    CONFIDENCE_THRESHOLD = 90
    if confidence >= CONFIDENCE_THRESHOLD:
        print("  [RAG] Skipped — model is confident enough")
        final = apply_elimination(first_pass_answer)   # still avoid known-wrong answers
        _last_decision = {"pass1": first_pass_answer, "pass2": first_pass_answer,
                          "overrode": final != first_pass_answer}
        return final

    print(f"  [RAG] Confidence too low ({confidence}% < {CONFIDENCE_THRESHOLD}%) — retrieving context...")
    context = retrieve_context(question_text, options, timeout=8.0)

    if not context:
        print("  [RAG] Nothing retrieved — sticking with Pass 1 answer")
        final = apply_elimination(first_pass_answer)
        _last_decision = {"pass1": first_pass_answer, "pass2": first_pass_answer,
                          "overrode": final != first_pass_answer}
        return final

    # ================= PASS 2: answer with context (2-vote) =================
    messages_pass2 = [
        {"role": "system", "content": (
            "You are a contestant on a quiz show. "
            "You will be given a short reference passage and a question with four options. "
            "The passage has been verified as directly relevant to the question. "
            "Use it to pick the best answer. "
            "Reply with ONLY one digit (0, 1, 2, or 3) and nothing else."
        )},
        {"role": "user", "content": (
            f"Reference passage:\n{context}\n\nQuestion: {question_text}\n"
            f"Options:\n{formatted}\n\nCorrect option digit:"
        )},
    ]
    inputs2 = tokenizer.apply_chat_template(
        messages_pass2, tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")

    pass2_answers = []
    for _ in range(2):
        with torch.inference_mode():
            out2 = model.generate(input_ids=inputs2, max_new_tokens=4, max_length=None,
                                   temperature=0.4, do_sample=True, use_cache=True)
        r2 = tokenizer.decode(out2[0][inputs2.shape[1]:], skip_special_tokens=True).strip()
        m2 = re.search(r"[0-3]", r2)
        if m2:
            pass2_answers.append(int(m2.group(0)))

    pass2_final = (pass2_answers[0]
                   if len(pass2_answers) == 2 and pass2_answers[0] == pass2_answers[1]
                   else first_pass_answer)
    print(f"  [Pass 2] votes={pass2_answers} -> {pass2_final}"
          + (f" (CHANGED from {first_pass_answer})" if pass2_final != first_pass_answer else " (kept)"))

    # Record decision for counterfactual analysis (before elimination)
    _last_decision = {"pass1": first_pass_answer, "pass2": pass2_final,
                      "overrode": pass2_final != first_pass_answer}

    # Apply elimination of known-wrong answers as a final safety net
    final_answer = apply_elimination(pass2_final)
    return final_answer


In [149]:
import json, os, time

# LOG_FILE is defined in the setup cell (it points to Drive)

def log_question(session_id, level, question_text, options, answer_given,
                 correct, decision=None, comp_id=None, mode="text"):
    # Write each answer into the log so we can learn from it later
    entry = {
        "session_id": session_id,
        "level": level,
        "comp_id": comp_id,
        "mode": mode,
        "config": RUN_CONFIG,
        "question": question_text,
        "options": [{"id": o.id, "text": o.text} for o in options],
        "answer_given": answer_given,
        "correct": correct,
    }
    if decision:
        entry["pass1"]    = decision.get("pass1")
        entry["pass2"]    = decision.get("pass2")
        entry["overrode"] = decision.get("overrode")
    log = []
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE) as f:
            log = json.load(f)
    log.append(entry)
    with open(LOG_FILE, "w") as f:
        json.dump(log, f, indent=2)


def play_game_automated(game, comp_id=None):
    # Play the game — supports both text mode and speech mode
    mode = getattr(game, "mode", "text")
    print(f"\n=== Bot is playing Session: {game.session_id}  (mode: {mode}) ===")

    while game.in_progress:
        question_obj = game.current_question
        if not question_obj:
            print("No question available. Ended, the game may have.")
            break

        # ---- Hear or read the question, depending on the mode ----
        if mode == "speech":
            # The question is spoken aloud — transcribe it
            question_text = transcribe_audio(game.fetch_audio_question(), comp_id)
            spoken_options = []
            for i in range(len(question_obj.options)):
                opt_text = transcribe_audio(
                    game.fetch_audio_option_next(), comp_id, context=question_text
                )
                spoken_options.append(opt_text)
            # Wrap transcribed options to look like the option objects the pipeline expects
            class _Opt:
                def __init__(self, id, text): self.id = id; self.text = text
            options = [_Opt(i, t) for i, t in enumerate(spoken_options)]
        else:
            # The question arrives as plain text — read it directly
            question_text = question_obj.text
            options = question_obj.options

        print(f"\n--- Level {game.current_level} ---")
        print(f"Q: {question_text}")
        for opt in options:
            print(f"  [{opt.id}] {opt.text}")

        # ---- Answer it via the pipeline (memory -> few-shot/RAG -> answer) ----
        print("Thinking...")
        answer_id = get_llm_answer(question_text, options, comp_id=comp_id)
        print(f"Bot chose: {answer_id}")

        result = game.answer(answer_id)
        time.sleep(1.5)              # avoid rapid consecutive requests (per the rules)

        log_question(
            session_id   = game.session_id,
            level        = game.current_level,
            question_text= question_text,
            options      = options,
            answer_given = answer_id,
            correct      = result.correct,
            decision     = _last_decision,
            comp_id      = comp_id,
            mode         = mode,
        )

        if result.correct:
            print(f"CORRECT! Earned: ${result.earned_amount:,.2f}")
            if result.game_over:
                print("CONGRATULATIONS! Cleared, the game is!")
        elif result.timed_out:
            print("TIMED OUT! Reached, the 30s limit was.")
            break
        else:
            print(f"WRONG! Over, the game is. Total: ${result.earned_amount:,.2f}")
            break


In [150]:
# (Drive already mounted in the setup cell above — LOG_FILE points to Drive)
# This cell intentionally left as a no-op to avoid a second conflicting mount.
print(f"Logging to: {LOG_FILE}")

Logging to: /content/gdrive/MyDrive/Colab Notebooks/game_log.json


## Ablation: zero-shot vs few-shot accuracy

Run this on a set of questions NOT in the training pool to measure the effect of
few-shot examples on genuinely unseen questions. This answers the assignment's
"what prompt (zero, few shot) gives the best results?" investigation question.

In [151]:
# ---- Few-shot ablation on held-out questions ----
# Reconstructs the same 15% held-out questions and tests N_FEWSHOT = 0, 3, 5.
import re, torch, json

def _nq(q):
    return re.sub(r"\s+", " ", q.strip().lower())

def answer_with_fewshot(question_text, options_list, n):
    formatted = "\n".join(f"[{i}] {t}" for i, t in enumerate(options_list))
    block = build_fewshot_block(question_text, n=n) if n > 0 else ""
    msgs = [
        {"role": "system", "content": SYSTEM_PASS1},
        {"role": "user", "content": (
            f"{block}Question: {question_text}\nOptions:\n{formatted}\n\n"
            f"Digit (0-3):\nConfidence (LOW/MEDIUM/HIGH):"
        )},
    ]
    ins = tokenizer.apply_chat_template(msgs, tokenize=True,
            add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(input_ids=ins, max_new_tokens=12, max_length=None,
                             temperature=0.3, do_sample=True, use_cache=True)
    resp = tokenizer.decode(out[0][ins.shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r"[0-3]", resp.split("\n")[0])
    return int(m.group(0)) if m else -1

# Reconstruct the held-out 15% (same dedup + tail split as the fine-tuning dataset)
with open(LOG_FILE) as f:
    _log = json.load(f)
seen, uniq = set(), []
for e in _log:
    if not e.get("correct"):
        continue
    qn = _nq(e["question"])
    if qn in seen:
        continue
    seen.add(qn)
    uniq.append(e)
held = uniq[int(len(uniq) * 0.85):]
print(f"Evaluating on {len(held)} held-out questions\n")

for n in [0, 3, 5]:
    correct = 0
    for e in held:
        opts = [o["text"] for o in sorted(e["options"], key=lambda x: x["id"])]
        pred = answer_with_fewshot(e["question"], opts, n)
        if pred == e["answer_given"]:
            correct += 1
    print(f"N_FEWSHOT={n}:  {correct}/{len(held)} = {correct/len(held)*100:.1f}%")


Evaluating on 115 held-out questions

N_FEWSHOT=0:  106/115 = 92.2%
N_FEWSHOT=3:  81/115 = 70.4%
N_FEWSHOT=5:  81/115 = 70.4%


## Fine-tuning the model on logged answers (LoRA)

This section uses the logged correct answers as a training set to fine-tune the model
with LoRA. Run it **once** to produce an adapter, then reload the adapter for gameplay.

**Important caveats**
- Train/validation split holds out 15% so we measure generalization, not memorization.
- Keep epochs low (2-3) and rank low (r=16) — the dataset is small and overfits easily.
- Only run this AFTER you have a meaningful number of logged correct answers.
- The base model already has `get_peft_model` applied (the LoRA scaffolding), so it is trainable.

In [152]:
# ---- Build the fine-tuning dataset from the log ----
import json, os, random, re
from datasets import Dataset

def _normq(q):
    return re.sub(r"\s+", " ", q.strip().lower())

with open(LOG_FILE) as f:
    _log = json.load(f)

# Keep correct answers, dedup by question
seen, pairs = set(), []
for e in _log:
    if not e.get("correct"):
        continue
    qn = _normq(e["question"])
    if qn in seen:
        continue
    seen.add(qn)
    opts = "\n".join(f"[{o['id']}] {o['text']}" for o in e["options"])
    prompt = (
        "You are a quiz show contestant. Choose the single best answer.\n"
        f"Question: {e['question']}\nOptions:\n{opts}\n"
        "Reply with ONLY the digit (0, 1, 2, or 3)."
    )
    pairs.append({"prompt": prompt, "answer": str(e["answer_given"])})

random.seed(3407)
random.shuffle(pairs)
split = int(len(pairs) * 0.85)
train_pairs, val_pairs = pairs[:split], pairs[split:]
print(f"Total unique correct Q/A: {len(pairs)}  |  train: {len(train_pairs)}  val: {len(val_pairs)}")

# Format into chat-template text for SFT
def to_text(rec):
    msgs = [
        {"role": "user", "content": rec["prompt"]},
        {"role": "assistant", "content": rec["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

train_ds = Dataset.from_list(train_pairs).map(to_text)
print(train_ds[0]["text"][:400])

Total unique correct Q/A: 763  |  train: 648  val: 115


Map:   0%|          | 0/648 [00:00<?, ? examples/s]

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
You are a quiz show contestant. Choose the single best answer.
Question: What term describes the concept that individuals have absolute ownership of their person and property in anarcho-capitalism?
Options:
[0] Sovereign state
[1] Mutual aid
[2] Collective ownership
[3] Individual sov


In [160]:
import gc, torch

# Free Whisper (not needed during training)
try:
    del whisper_model
except NameError:
    pass

# Free any leftover trainer from a previous attempt
try:
    del trainer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

# Check what's free now
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

GPU free: 2.85 GB / 15.64 GB


In [161]:
# ---- Train with Unsloth SFTTrainer ----
from trl import SFTTrainer
from transformers import TrainingArguments

FastLanguageModel.for_training(model)   # switch back to training mode

trainer = SFTTrainer(
    model               = model,
    tokenizer           = tokenizer,
    train_dataset       = train_ds,
    dataset_text_field  = "text",
    max_seq_length      = 256,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps      = 5,
        num_train_epochs  = 2,          # keep low — small dataset overfits fast
        learning_rate     = 2e-4,
        fp16              = not torch.cuda.is_bf16_supported(),
        bf16              = torch.cuda.is_bf16_supported(),
        logging_steps     = 5,
        optim             = "adamw_8bit",
        weight_decay      = 0.01,
        lr_scheduler_type = "linear",
        seed              = 3407,
        output_dir        = "outputs",
        report_to         = "none",
        gradient_checkpointing      = True,   # trades compute for memory
    ),
)
trainer.train()
FastLanguageModel.for_inference(model)   # back to inference mode after training

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/648 [00:00<?, ? examples/s]

{'loss': '0.7293', 'grad_norm': '0.382', 'learning_rate': '0.00016', 'epoch': '0.03086'}
{'loss': '0.6339', 'grad_norm': '0.3184', 'learning_rate': '0.0001975', 'epoch': '0.06173'}
{'loss': '0.6335', 'grad_norm': '0.352', 'learning_rate': '0.0001944', 'epoch': '0.09259'}
{'loss': '0.6629', 'grad_norm': '0.3712', 'learning_rate': '0.0001912', 'epoch': '0.1235'}
{'loss': '0.6396', 'grad_norm': '0.4245', 'learning_rate': '0.0001881', 'epoch': '0.1543'}
{'loss': '0.6579', 'grad_norm': '0.3757', 'learning_rate': '0.000185', 'epoch': '0.1852'}
{'loss': '0.6338', 'grad_norm': '0.3704', 'learning_rate': '0.0001818', 'epoch': '0.216'}
{'loss': '0.5761', 'grad_norm': '0.384', 'learning_rate': '0.0001787', 'epoch': '0.2469'}
{'loss': '0.5433', 'grad_norm': '0.5553', 'learning_rate': '0.0001755', 'epoch': '0.2778'}
{'loss': '0.6681', 'grad_norm': '0.3993', 'learning_rate': '0.0001724', 'epoch': '0.3086'}
{'loss': '0.6156', 'grad_norm': '0.4186', 'learning_rate': '0.0001693', 'epoch': '0.3395'}
{'l

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 5120, padding_idx=151665)
        (layers): ModuleList(
          (0-47): 48 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=5120, out_features=5120, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=5120, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=5120, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [162]:
# ---- Validate on the held-out set (measures GENERALIZATION) ----
import re, torch

def quick_answer(prompt):
    msgs = [{"role": "user", "content": prompt}]
    ins = tokenizer.apply_chat_template(msgs, tokenize=True,
            add_generation_prompt=True, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        out = model.generate(input_ids=ins, max_new_tokens=4, max_length=None,
                             temperature=0.1, do_sample=True, use_cache=True)
    resp = tokenizer.decode(out[0][ins.shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r"[0-3]", resp)
    return int(m.group(0)) if m else -1

correct = 0
for rec in val_pairs:
    pred = quick_answer(rec["prompt"])
    if pred == int(rec["answer"]):
        correct += 1
print(f"Held-out accuracy after fine-tuning: {correct}/{len(val_pairs)} = {correct/len(val_pairs)*100:.1f}%")
print("(Compare this to the base model's accuracy on the same val_pairs to see if fine-tuning helped.)")

Held-out accuracy after fine-tuning: 110/115 = 95.7%
(Compare this to the base model's accuracy on the same val_pairs to see if fine-tuning helped.)


In [174]:
# ---- Save the LoRA adapter to Drive so you don't retrain every session ----
ADAPTER_DIR = os.path.join(DRIVE_DIR, "polimillionaire_lora") if "DRIVE_DIR" in dir() else "polimillionaire_lora"
# model.save_pretrained(ADAPTER_DIR)
# tokenizer.save_pretrained(ADAPTER_DIR)
# print(f"Adapter saved to {ADAPTER_DIR}")

# To reload later (in a fresh session, after loading the base model):
from peft import PeftModel
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
FastLanguageModel.for_inference(model)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): Qwen2ForCausalLM(
          (model): Qwen2Model(
            (embed_tokens): Embedding(152064, 5120, padding_idx=151665)
            (layers): ModuleList(
              (0-47): 48 x Qwen2DecoderLayer(
                (self_attn): Qwen2Attention(
                  (q_proj): lora.Linear4bit(
                    (base_layer): Linear4bit(in_features=5120, out_features=5120, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Identity()
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=5120, out_features=16, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=16, out_features=5120, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
            

In [167]:
from faster_whisper import WhisperModel
WHISPER_MODEL = "deepdml/faster-whisper-large-v3-turbo-ct2"
whisper_model = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="int8")

In [ ]:
# ============================================================
# START THE GAME — choose your mode
#   MODE = "text"   -> the agent reads the questions
#   MODE = "speech" -> the agent hears the questions aloud
# ============================================================
import os, sys

# Silence kernel-level warnings (redirect stderr)
class _SuppressStderr:
    def __enter__(self):
        self._saved = os.dup(2)                       # save real stderr fd
        self._devnull = os.open(os.devnull, os.O_WRONLY)
        os.dup2(self._devnull, 2)                      # redirect fd 2 to devnull
        return self
    def __exit__(self, *a):
        os.dup2(self._saved, 2)                        # restore
        os.close(self._devnull)
        os.close(self._saved)

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

MODE = "text"     # "text" or "speech" — change this, you may

print(f"\n=== Starting Game (mode: {MODE}) ===")
try:
    game = client.game.start(competition_id=comp_id, mode=MODE)
except TypeError:
    # The server may not support the mode argument — fall back to text
    game = client.game.start(competition_id=comp_id)
    print("Mode argument unsupported — in text mode, play we will.")

print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}\n")

with _SuppressStderr():
    play_game_automated(game, comp_id=comp_id)



=== Starting Game (mode: text) ===
Session ID: 367351
Total number of questions: 15


=== Bot is playing Session: 367351  (mode: text) ===

--- Level 1 ---
Q: What is the fundamental principle by which ancient Egyptian citizens lived?
  [0] Karma
  [1] Democracy
  [2] Maat
  [3] Fate
Thinking...
  [Memory] Previously correct: 'maat' — using option 2
Bot chose: 2
CORRECT! Earned: $100.00

--- Level 2 ---
Q: How does the concept of democracy in ancient Greece differ from modern democracy?
  [0] Ancient Greeks had a direct democracy where all citizens could participate, while modern democracy often involves representative government
  [1] Ancient Greeks and modern democracies are identical in their structures
  [2] Ancient Greeks had a representative democracy, while modern democracy has direct participation
  [3] There was no concept of democracy in ancient Greece
Thinking...
  [Pass 1 raw]: 0
85
  [Pass 1] Answer: 0  Confidence: 85%
  [RAG] Confidence too low (85% < 90%) — retrieving c

/tmp/ipykernel_8906/965343564.py:145: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


  [RAG] [Web – totalshape.com] — 1.9s (204 chars) relevance=-0.07
  [RAG] Relevance too low — discarding context
  [RAG] Nothing retrieved — sticking with Pass 1 answer
Bot chose: 0


In [181]:
import json

# LOG_FILE defined in setup cell (Drive path)

with open(LOG_FILE) as f:
    log = json.load(f)
overrides = [e for e in log if e.get("overrode")]
helped = sum(1 for e in overrides if e["correct"])
hurt   = sum(1 for e in overrides if not e["correct"])
print(f"Overrides: {len(overrides)} | helped: {helped} | hurt: {hurt}")
if overrides:
    print(f"Override accuracy: {helped/len(overrides)*100:.0f}%")

Overrides: 63 | helped: 31 | hurt: 32
Override accuracy: 49%


## Performance — by category and by mode

For each competition and each mode (text vs speech), we compute the **average level reached** across sessions and the **maximum level reached**. The level reached in a session is the highest level its questions touched. This gives a clean comparison of text against speech, which is valuable for the report.

In [202]:
# ============================================================
# PER-CATEGORY, PER-MODE PERFORMANCE
#   Average level reached + maximum level reached, per (category, mode).
#   A "session" = one game; its level reached = the max level it touched.
# ============================================================
import json
from collections import defaultdict

COMP_NAMES = {0: "Entertainment", 1: "Ancient History", 2: "Science & Nature",
              3: "Maths", 4: "Philosophy & Psychology", 5: "News"}

with open(LOG_FILE) as f:
    _log = json.load(f)

# Group questions by (comp_id, mode, session) — find the max level per session
# Treat old entries without 'mode' as "text".
sessions = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
# sessions[comp_id][mode][session_id] = max level reached in that session

for e in _log:
    cid  = e.get("comp_id")
    mode = e.get("mode", "text")
    sid  = e.get("session_id")
    lvl  = e.get("level", 0)
    if cid is None or sid is None:
        continue
    if lvl > sessions[cid][mode][sid]:
        sessions[cid][mode][sid] = lvl

# Also compute per-question accuracy
acc = defaultdict(lambda: defaultdict(lambda: [0, 0]))   # [correct, total]
for e in _log:
    cid  = e.get("comp_id")
    mode = e.get("mode", "text")
    if cid is None:
        continue
    acc[cid][mode][1] += 1
    if e.get("correct"):
        acc[cid][mode][0] += 1

print("=" * 78)
print(f"{'Category':<24}{'Mode':<9}{'Sessions':>9}{'Avg Lvl':>9}{'Max Lvl':>9}{'Accuracy':>11}")
print("=" * 78)

for cid in sorted(sessions.keys()):
    name = COMP_NAMES.get(cid, f"Comp {cid}")
    for mode in sorted(sessions[cid].keys()):
        reached = list(sessions[cid][mode].values())
        n_sess  = len(reached)
        avg_lvl = sum(reached) / n_sess if n_sess else 0
        max_lvl = max(reached) if reached else 0
        c, t    = acc[cid][mode]
        acc_pct = (c / t * 100) if t else 0
        print(f"{name:<24}{mode:<9}{n_sess:>9}{avg_lvl:>9.1f}{max_lvl:>9}{acc_pct:>10.1f}%")

print("=" * 78)
print("Avg Lvl = mean level reached across games | Max Lvl = best single game")
print("Accuracy = correct answers / total questions answered")

# STT timing summary (speech mode only), if any speech was played
if 'stt_timings' in dir() and stt_timings:
    import numpy as np
    print(f"\nSpeech transcription time: avg {np.mean(stt_timings):.2f}s | "
          f"max {np.max(stt_timings):.2f}s | n={len(stt_timings)}")


Category                Mode      Sessions  Avg Lvl  Max Lvl   Accuracy
Ancient History         speech           8      2.2        4      55.6%
Ancient History         text            25      9.1       15      91.9%
Science & Nature        speech          10      5.6       15      83.9%
Science & Nature        text            14      6.3       15      91.6%
Philosophy & Psychology speech           8      5.9       15      83.0%
Philosophy & Psychology text             8      7.0       15      98.0%
News                    text             3      2.0        2      60.0%
Avg Lvl = mean level reached across games | Max Lvl = best single game
Accuracy = correct answers / total questions answered

Speech transcription time: avg 0.41s | max 0.50s | n=5


## History A/B — fine-tuning + few-shot vs base model + RAG

This cell measures the gain of the adapter and few-shot for the history category alone. Two configurations were played separately and tagged in the log:

- **`base+rag`** — the plain base model with RAG only (no adapter, no few-shot)
- **`adapter+fewshot`** — the fine-tuned LoRA adapter plus few-shot prompting

For each configuration, we report the average level reached, the maximum level reached, and the accuracy. This is a fair comparison: both runs were played with `USE_MEMORY = False`, so the answer cache could not contaminate it. The table reveals the true contribution of fine-tuning and few-shot over the base model.

In [203]:
import json
from collections import defaultdict

with open(LOG_FILE) as f:
    _log = json.load(f)

hist = [e for e in _log if e.get("comp_id") == 1 and "config" in e]
by_cfg = defaultdict(lambda: defaultdict(int))
acc = defaultdict(lambda: [0, 0])
for e in hist:
    cfg, sid = e["config"], e["session_id"]
    by_cfg[cfg][sid] = max(by_cfg[cfg][sid], e.get("level", 0))
    acc[cfg][1] += 1
    if e.get("correct"): acc[cfg][0] += 1

print(f"{'Config':<20}{'Games':>7}{'Avg Lvl':>9}{'Max Lvl':>9}{'Acc':>9}")
for cfg in sorted(by_cfg):
    r = list(by_cfg[cfg].values())
    c, t = acc[cfg]
    print(f"{cfg:<20}{len(r):>7}{sum(r)/len(r):>9.1f}{max(r):>9}{c/t*100:>8.1f}%")

Config                Games  Avg Lvl  Max Lvl      Acc
adapter+fewshot          12      7.3       15    87.4%
model+rag                 7     11.1       15    94.7%
